# Phase 1: data split + ROI sanity check (DICOM input)
Run on Kaggle (Internet ON). Steps: clone/pull repo -> install DICOM decoders -> patient-level split -> look at 20 ROI crops.

In [ ]:
import os
if not os.path.isdir("mammogram-cancer-classifier"):
    !git clone https://github.com/hassenmohamed-mezr/mammogram-cancer-classifier.git
%cd /kaggle/working/mammogram-cancer-classifier
!git pull
!pip -q install pydicom pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg
# if DICOM decoding still fails later, also try:  !pip -q install python-gdcm

In [ ]:
# 1) look at what is attached
!ls /kaggle/input
!ls /kaggle/input/rsna-breast-cancer-detection
!ls /kaggle/input/rsna-breast-cancer-detection/train_images | head -3

In [ ]:
# ---- EDIT only if the ls above shows different names ----
COMP     = "/kaggle/input/rsna-breast-cancer-detection"
CSV      = f"{COMP}/train.csv"
IMG_ROOT = f"{COMP}/train_images"
IMG_TMPL = "{root}/{patient_id}/{image_id}.dcm"
OUT      = "/kaggle/working/splits"

In [ ]:
# eval-neg-ratio 10: caps negatives per positive in val/test to save ROI time
# (AUC unaffected; PR-AUC is inflated -> mention it in the results)
!python -m src.data --csv {CSV} --img-root {IMG_ROOT} --img-template "{IMG_TMPL}" --out {OUT} --ratios 1 3 --eval-neg-ratio 10

In [ ]:
import pandas as pd
tr = pd.read_csv(f"{OUT}/train_3to1.csv"); va = pd.read_csv(f"{OUT}/val.csv"); te = pd.read_csv(f"{OUT}/test.csv")

# no patient leakage between splits
assert not (set(tr.patient_id) & set(va.patient_id)) and not (set(tr.patient_id) & set(te.patient_id)) and not (set(va.patient_id) & set(te.patient_id))

# paths exist
sample = tr.sample(10, random_state=0)
missing = [p for p in sample.path if not os.path.exists(p)]
print("missing paths:", missing[:3], "->", "OK" if not missing else "FIX IMG_TMPL / IMG_ROOT")

# one DICOM decodes correctly
from src.roi import load_image
x = load_image(sample.path.iloc[0])
print("decoded:", x.shape, x.dtype, x.min(), x.max())

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.roi import crop_roi, load_image

# Plug the winner's YOLOX here (from your rsna_inference notebook):
#   detector(img_uint8) -> (x1, y1, x2, y2, conf)  or  None
detector = None   # None = fallback (Otsu) only, so the pipeline still runs

pos = tr[tr.cancer == 1].sample(10, random_state=1)
neg = tr[tr.cancer == 0].sample(10, random_state=1)
rows = pd.concat([pos, neg])

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
    img = load_image(r.path)
    crop, info = crop_roi(img, detector=detector)
    ax.imshow(crop, cmap="gray"); ax.axis("off")
    ax.set_title(f"cancer={r.cancer} | {info['source']} | {crop.shape[1]}x{crop.shape[0]}", fontsize=9)
plt.tight_layout(); plt.show()

**Look at the grid:** is the whole breast inside every crop (no big black area, no cut-off tissue, not inverted)?
If yes -> next step: run the ROI on `roi_manifest.csv`. If not -> describe what you see.